# 🧠 Always-On Memory Agent — llama.cpp 版 (Qwen2.5 GGUF / CPU)

このノートブックは [GoogleCloudPlatform/generative-ai の always-on-memory-agent](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/gemini/agents/always-on-memory-agent) を、
**llama-cpp-python + Qwen2.5 GGUF** で動かせるよう改変したものです。

## Ollama 版との違い

| 項目 | Ollama 版 | **この llama.cpp 版** |
|------|-----------|-----------------------|
| バックエンド | Ollama (Go + llama.cpp) | **llama-cpp-python (純Python バインディング)** |
| インストール | curl スクリプト | **pip のみ** |
| モデル取得 | `ollama pull` | **HuggingFace から直接 GGUF** |
| API 互換 | Ollama API | **OpenAI 互換 API (port 8000)** |
| CPU 最適化 | Ollama 任せ | **スレッド数・量子化を細かく制御** |
| 依存 | Ollama バイナリ必須 | **pip install のみ・Ollama 不要** |

## スタック構成
```
┌──────────────────────────────────────────────────────────┐
│              Memory Agent HTTP API (port 8888)            │
│   IngestAgent │ QueryAgent │ ConsolidateAgent (ADK)       │
│                     ↕ LiteLLM                            │
│   llama-cpp-python サーバー  (OpenAI互換 port 8000)       │
│          Qwen2.5-1.5B-Instruct-GGUF (Q4_K_M)             │
│                  ↑ HuggingFace Hub からダウンロード        │
└──────────────────────────────────────────────────────────┘
```

> **💡 Colab での注意**: ランタイムは CPU で動きます。T4 GPU ランタイムにすると `n_gpu_layers` を設定してさらに高速化できます。

## ① 依存パッケージのインストール

`llama-cpp-python[server]` に OpenAI 互換サーバーが含まれています。  
CPU ビルド（デフォルト）なので特別なコンパイルオプションは不要です。

In [ ]:
%%bash
# llama-cpp-python[server]: OpenAI互換サーバー込みのインストール
# CPU向けのデフォルトビルド (CMAKE_ARGS 不要)
pip install -q "llama-cpp-python[server]" huggingface_hub \
               google-adk litellm aiohttp pyngrok 2>&1 | tail -8
echo "✅ インストール完了"

## ② モデルの設定とダウンロード

HuggingFace Hub から **Qwen2.5-1.5B-Instruct の GGUF (Q4_K_M量子化)** を直接ダウンロードします。  
約 **1GB** で CPU でも快適に動作します。

| モデル | サイズ | 推奨環境 | HF repo |
|--------|--------|----------|---------|
| `Qwen2.5-1.5B Q4_K_M` | ~1GB | **CPU（推奨・デフォルト）** | `Qwen/Qwen2.5-1.5B-Instruct-GGUF` |
| `Qwen2.5-3B Q4_K_M` | ~2GB | CPU (RAM 8GB以上) | `Qwen/Qwen2.5-3B-Instruct-GGUF` |
| `Qwen2.5-7B Q4_K_M` | ~5GB | T4 GPU / RAM 16GB | `Qwen/Qwen2.5-7B-Instruct-GGUF` |

In [ ]:
from huggingface_hub import hf_hub_download
import os, time

# ---- モデル設定 (変更可能) ----
HF_REPO     = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
GGUF_FILE   = "qwen2.5-1.5b-instruct-q4_k_m.gguf"  # Q4_K_M: CPU向けバランス量子化
# 他の選択肢:
#   GGUF_FILE = "qwen2.5-1.5b-instruct-q8_0.gguf"   # Q8_0: より高精度 (~1.6GB)
#   GGUF_FILE = "qwen2.5-1.5b-instruct-q2_k.gguf"   # Q2_K: 最軽量 (~0.6GB)
CPU_THREADS = 4   # Colab CPU コア数に合わせる（通常2〜4）
N_CTX       = 2048  # コンテキスト長（CPU省メモリ向け）
# --------------------------------

print(f"⬇️  {HF_REPO} / {GGUF_FILE} をダウンロード中...")
t0 = time.time()
model_path = hf_hub_download(
    repo_id=HF_REPO,
    filename=GGUF_FILE,
    local_dir="/content/models",
    local_dir_use_symlinks=False,
)
elapsed = time.time() - t0
size_mb = os.path.getsize(model_path) / 1024**2
print(f"✅ ダウンロード完了: {model_path}")
print(f"   サイズ: {size_mb:.0f} MB  |  所要時間: {elapsed:.0f}秒")

## ③ llama-cpp-python サーバーの起動

`python -m llama_cpp.server` で **OpenAI 互換 API サーバー (port 8000)** を起動します。  
ADK は LiteLLM 経由でこのサーバーに接続します。

## 3層メモリアーキテクチャ

```
┌─────────────────────────────────────────────────────────────┐
│  Layer 3 : Insight（洞察）                                   │
│    複数の統合記憶をさらに抽象化した高次の知識                  │
│    例）「全体として○○という傾向がある」                       │
│                        ↑ InsightAgent が生成                │
│  Layer 2 : Consolidation（統合記憶）                         │
│    複数の生記憶をまとめ、関係性を見つけた中間知識              │
│    例）「記憶A・B・Cは○○で繋がっている」                     │
│                        ↑ ConsolidateAgent が生成            │
│  Layer 1 : Raw Memory（生記憶）                              │
│    IngestAgent が取り込んだ元テキスト＋構造化メタデータ        │
└─────────────────────────────────────────────────────────────┘
                  ↕  QueryAgent が3層すべてを参照
```

QueryAgent は **L3 → L2 → L1 の順に優先度をつけて**回答を合成します。

In [ ]:
import subprocess, time, requests, os

LLAMA_PORT = 8000

server_env = os.environ.copy()
# llama-cpp-python サーバー設定を環境変数で渡す
server_env["MODEL"]         = model_path
server_env["HOST"]          = "0.0.0.0"
server_env["PORT"]          = str(LLAMA_PORT)
server_env["N_CTX"]         = str(N_CTX)
server_env["N_THREADS"]     = str(CPU_THREADS)
server_env["CHAT_FORMAT"]   = "chatml"          # Qwen2.5 は chatml フォーマット
server_env["N_GPU_LAYERS"]  = "0"               # CPU のみ（GPU有りは 32〜99 に変更）
server_env["VERBOSE"]       = "false"

print(f"🚀 llama-cpp-python サーバー起動中 (port {LLAMA_PORT})...")
llama_proc = subprocess.Popen(
    ["python", "-m", "llama_cpp.server"],
    env=server_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# 起動確認（モデルロードに少し時間がかかる）
for i in range(20):
    try:
        r = requests.get(f"http://localhost:{LLAMA_PORT}/v1/models", timeout=3)
        models = r.json().get("data", [])
        print(f"\n✅ llama-cpp-python サーバー起動完了!")
        print(f"   利用可能モデル: {[m['id'] for m in models]}")
        break
    except:
        print(f"   モデルロード中... ({i+1}/20)", end="\r")
        time.sleep(4)
else:
    out = llama_proc.stdout.read(3000)
    print(f"\n❌ 起動失敗。ログ:\n{out}")

In [ ]:
# 簡易動作確認
print("🧪 推論テスト中...")
t0 = time.time()
r = requests.post(
    f"http://localhost:{LLAMA_PORT}/v1/chat/completions",
    json={
        "model": "qwen",
        "messages": [{"role": "user", "content": "日本の首都は？一言で答えよ。"}],
        "max_tokens": 20,
        "temperature": 0.1,
    },
    timeout=60,
)
elapsed = time.time() - t0
if r.ok:
    answer = r.json()["choices"][0]["message"]["content"].strip()
    print(f"   回答: {answer}")
    print(f"   ⏱️  応答時間: {elapsed:.1f}秒")
    print(f"   {'✅ CPU推論 OK' if elapsed < 30 else '⚠️ やや遅め（正常動作）'}")
else:
    print(f"⚠️ テスト失敗: {r.text}")

## ④ エージェントコードの生成

LiteLLM で llama-cpp-python の OpenAI 互換 API に接続します。  
`openai/` プレフィックス + `api_base` を指定するだけで OK です。

In [ ]:
%%writefile agent_llamacpp.py
import asyncio, sqlite3, json, re, os
from datetime import datetime
from aiohttp import web
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

LLAMA_API_BASE       = os.environ.get('LLAMA_API_BASE',        'http://localhost:8000/v1')
DB_PATH              = os.environ.get('MEMORY_DB_PATH',         'memory.db')
CONSOLIDATE_INTERVAL = int(os.environ.get('CONSOLIDATE_INTERVAL', '600'))
INSIGHT_INTERVAL     = int(os.environ.get('INSIGHT_INTERVAL',    '1800'))
API_PORT             = int(os.environ.get('API_PORT',            '8888'))
MAX_CONTENT_LEN      = 800
MAX_L1_QUERY         = 15
MAX_L2_QUERY         = 10
MAX_L3_QUERY         = 5


def get_llm():
    return LiteLlm(
        model='openai/qwen',
        api_base=LLAMA_API_BASE,
        api_key='sk-no-key',
        timeout=120,
        temperature=0.3,
        max_tokens=512,
        extra_body={'system': '必ず日本語で回答せよ。You must always respond in Japanese.'},
    )


def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        "CREATE TABLE IF NOT EXISTS memories ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT,"
        "content TEXT NOT NULL,"
        "source TEXT DEFAULT 'manual',"
        "summary TEXT,"
        "entities TEXT,"
        "topics TEXT,"
        "importance REAL DEFAULT 0.5,"
        "consolidated INTEGER DEFAULT 0,"
        "created_at REAL DEFAULT (strftime('%s','now')))"
    )
    c.execute(
        "CREATE TABLE IF NOT EXISTS consolidations ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT,"
        "layer INTEGER DEFAULT 2,"
        "source_ids TEXT,"
        "connections TEXT,"
        "insight TEXT,"
        "abstracted INTEGER DEFAULT 0,"
        "created_at REAL DEFAULT (strftime('%s','now')))"
    )
    conn.commit(); conn.close()


def db_save_memory(content, source, summary, entities, topics, importance):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        'INSERT INTO memories (content,source,summary,entities,topics,importance) VALUES(?,?,?,?,?,?)',
        (content, source, summary,
         json.dumps(entities, ensure_ascii=False),
         json.dumps(topics,   ensure_ascii=False), importance)
    )
    mid = c.lastrowid
    conn.commit(); conn.close()
    return mid


def db_get_l1(limit=100):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        'SELECT id,content,source,summary,entities,topics,importance,consolidated,created_at '
        'FROM memories ORDER BY importance DESC, created_at DESC LIMIT ?', (limit,)
    )
    rows = c.fetchall(); conn.close()
    return [{
        'id': r[0], 'content': r[1], 'source': r[2], 'summary': r[3],
        'entities': json.loads(r[4] or '[]'), 'topics': json.loads(r[5] or '[]'),
        'importance': r[6], 'consolidated': bool(r[7]),
        'created_at': datetime.fromtimestamp(r[8]).isoformat() if r[8] else None
    } for r in rows]


def db_get_l1_unconsolidated():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('SELECT id,content,summary FROM memories WHERE consolidated=0 ORDER BY created_at')
    rows = c.fetchall(); conn.close()
    return [{'id': r[0], 'content': r[1], 'summary': r[2]} for r in rows]


def db_mark_l1_consolidated(ids):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.executemany('UPDATE memories SET consolidated=1 WHERE id=?', [(i,) for i in ids])
    conn.commit(); conn.close()


def db_save_l2(source_ids, connections, insight):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        'INSERT INTO consolidations (layer,source_ids,connections,insight) VALUES(2,?,?,?)',
        (json.dumps(source_ids), connections, insight)
    )
    cid = c.lastrowid
    conn.commit(); conn.close()
    return cid


def db_get_l2(limit=50):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        'SELECT id,source_ids,connections,insight,abstracted,created_at '
        'FROM consolidations WHERE layer=2 ORDER BY created_at DESC LIMIT ?', (limit,)
    )
    rows = c.fetchall(); conn.close()
    return [{
        'id': r[0], 'source_ids': json.loads(r[1] or '[]'),
        'connections': r[2], 'insight': r[3],
        'abstracted': bool(r[4]),
        'created_at': datetime.fromtimestamp(r[5]).isoformat() if r[5] else None
    } for r in rows]


def db_get_l2_unabstracted():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        'SELECT id,connections,insight FROM consolidations '
        'WHERE layer=2 AND abstracted=0 ORDER BY created_at'
    )
    rows = c.fetchall(); conn.close()
    return [{'id': r[0], 'connections': r[1], 'insight': r[2]} for r in rows]


def db_mark_l2_abstracted(ids):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.executemany('UPDATE consolidations SET abstracted=1 WHERE id=?', [(i,) for i in ids])
    conn.commit(); conn.close()


def db_save_l3(source_ids, connections, insight):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        'INSERT INTO consolidations (layer,source_ids,connections,insight) VALUES(3,?,?,?)',
        (json.dumps(source_ids), connections, insight)
    )
    cid = c.lastrowid
    conn.commit(); conn.close()
    return cid


def db_get_l3(limit=20):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute(
        'SELECT id,source_ids,connections,insight,created_at '
        'FROM consolidations WHERE layer=3 ORDER BY created_at DESC LIMIT ?', (limit,)
    )
    rows = c.fetchall(); conn.close()
    return [{
        'id': r[0], 'source_ids': json.loads(r[1] or '[]'),
        'connections': r[2], 'insight': r[3],
        'created_at': datetime.fromtimestamp(r[4]).isoformat() if r[4] else None
    } for r in rows]


def db_delete_memory(mid):
    conn = sqlite3.connect(DB_PATH)
    conn.cursor().execute('DELETE FROM memories WHERE id=?', (mid,))
    conn.commit(); conn.close()


def db_clear_all():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('DELETE FROM memories')
    c.execute('DELETE FROM consolidations')
    conn.commit(); conn.close()


def db_get_stats():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('SELECT COUNT(*) FROM memories');                                  l1_total = c.fetchone()[0]
    c.execute('SELECT COUNT(*) FROM memories WHERE consolidated=0');             l1_new   = c.fetchone()[0]
    c.execute('SELECT COUNT(*) FROM consolidations WHERE layer=2');              l2_total = c.fetchone()[0]
    c.execute('SELECT COUNT(*) FROM consolidations WHERE layer=2 AND abstracted=0'); l2_new = c.fetchone()[0]
    c.execute('SELECT COUNT(*) FROM consolidations WHERE layer=3');              l3_total = c.fetchone()[0]
    conn.close()
    return {
        'L1_raw_memories':   {'total': l1_total, 'unconsolidated': l1_new},
        'L2_consolidations': {'total': l2_total, 'unabstracted':   l2_new},
        'L3_insights':       {'total': l3_total},
    }


async def run_agent(agent, session_service, message):
    session = await session_service.create_session(app_name='memory_agent', user_id='system')
    runner  = Runner(agent=agent, app_name='memory_agent', session_service=session_service)
    content = genai_types.Content(role='user', parts=[genai_types.Part(text=message)])
    text    = ''
    async for event in runner.run_async(user_id='system', session_id=session.id, new_message=content):
        if hasattr(event, 'content') and event.content:
            for part in event.content.parts:
                if hasattr(part, 'text') and part.text:
                    text += part.text
    return text.strip()


def extract_json(text):
    m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if m:
        try:    return json.loads(m.group())
        except: pass
    return None


INGEST_INSTRUCTION = (
    '必ず日本語で回答せよ。\n'
    'テキストを分析し、以下の JSON のみを返せ（コードブロック不要）:\n'
    '{"summary":"要約(30字以内)","entities":[],"topics":[],"importance":0.5}'
)

async def ingest_text(text, source='manual'):
    truncated = text[:MAX_CONTENT_LEN]
    session_service = InMemorySessionService()
    agent = Agent(name='ingest_agent', model=get_llm(), instruction=INGEST_INSTRUCTION)
    try:
        raw  = await run_agent(agent, session_service, '分析:\n' + truncated)
        data = extract_json(raw) or {}
    except Exception as e:
        print('  [ingest warn]', e)
        data = {}
    mid = db_save_memory(
        content=text, source=source,
        summary=data.get('summary', text[:80]),
        entities=data.get('entities', []),
        topics=data.get('topics', []),
        importance=float(data.get('importance', 0.5)),
    )
    return {'layer': 1, 'memory_id': mid, **data}


CONSOLIDATE_INSTRUCTION = (
    '必ず日本語で回答せよ。\n'
    '複数の記憶を分析し、記憶間の関係性を見つけよ。\n'
    '以下の JSON のみを返せ（コードブロック不要）:\n'
    '{"connections":"記憶間の関係性（具体的に）","insight":"共通テーマや重要な気づき"}'
)

async def consolidate_l1():
    batch = db_get_l1_unconsolidated()[:10]
    if len(batch) < 2:
        return {'status': 'skipped', 'reason': 'L1未統合記憶が2件未満'}
    mem_ctx = '\n'.join([
        '[L1-' + str(m['id']) + '] ' + (m['summary'] or m['content'])[:120]
        for m in batch
    ])
    session_service = InMemorySessionService()
    agent = Agent(name='consolidate_agent', model=get_llm(), instruction=CONSOLIDATE_INSTRUCTION)
    try:
        raw  = await run_agent(agent, session_service, '記憶群:\n' + mem_ctx)
        data = extract_json(raw) or {'connections': raw[:200], 'insight': ''}
    except Exception:
        data = {'connections': '解析エラー', 'insight': ''}
    ids = [m['id'] for m in batch]
    cid = db_save_l2(ids, data.get('connections', ''), data.get('insight', ''))
    db_mark_l1_consolidated(ids)
    return {'status': 'ok', 'layer': 2, 'consolidation_id': cid,
            'consolidated_l1_count': len(ids), **data}


INSIGHT_INSTRUCTION = (
    '必ず日本語で回答せよ。\n'
    '複数の統合記憶（L2）をさらに高次に抽象化し、全体を貫く原則や傾向を見つけよ。\n'
    '以下の JSON のみを返せ（コードブロック不要）:\n'
    '{"connections":"L2統合間の共通パターン","insight":"全体を貫く高次の洞察・原則"}'
)

async def abstract_l2():
    batch = db_get_l2_unabstracted()[:8]
    if len(batch) < 2:
        return {'status': 'skipped', 'reason': 'L2未抽象化統合が2件未満'}
    l2_ctx = '\n'.join([
        '[L2-' + str(m['id']) + '] 関係性: ' + m['connections'][:100] + ' / 気づき: ' + m['insight'][:80]
        for m in batch
    ])
    session_service = InMemorySessionService()
    agent = Agent(name='insight_agent', model=get_llm(), instruction=INSIGHT_INSTRUCTION)
    try:
        raw  = await run_agent(agent, session_service, '統合記憶群（L2）:\n' + l2_ctx)
        data = extract_json(raw) or {'connections': raw[:200], 'insight': ''}
    except Exception:
        data = {'connections': '解析エラー', 'insight': ''}
    ids = [m['id'] for m in batch]
    cid = db_save_l3(ids, data.get('connections', ''), data.get('insight', ''))
    db_mark_l2_abstracted(ids)
    return {'status': 'ok', 'layer': 3, 'insight_id': cid,
            'abstracted_l2_count': len(ids), **data}


QUERY_INSTRUCTION = (
    '必ず日本語で回答せよ。'
    '3層の記憶（L3高次洞察・L2統合記憶・L1生記憶）を優先度に従って参照し、'
    '根拠を示しながら質問に答えよ。'
)

async def query_memory(question):
    l1 = db_get_l1(limit=MAX_L1_QUERY)
    l2 = db_get_l2(limit=MAX_L2_QUERY)
    l3 = db_get_l3(limit=MAX_L3_QUERY)
    if not l1 and not l2 and not l3:
        return 'まだ記憶がありません。/ingest でテキストを追加してください。'
    l3_ctx = '\n'.join(['[L3-' + str(m['id']) + '] ' + m['insight'][:200] for m in l3]) if l3 else '（洞察なし）'
    l2_ctx = '\n'.join(['[L2-' + str(m['id']) + '] 関係性: ' + m['connections'][:120] + ' / 気づき: ' + m['insight'][:100] for m in l2]) if l2 else '（統合記憶なし）'
    l1_ctx = '\n'.join(['[L1-' + str(m['id']) + '] ' + (m['summary'] or m['content'])[:120] for m in l1])
    prompt = (
        '以下の3層の記憶を参照して質問に答えよ。\n'
        '優先度: L3（高次洞察）> L2（統合記憶）> L1（生記憶）\n'
        '回答時は使用した記憶を [L3-ID], [L2-ID], [L1-ID] の形式で引用すること。\n\n'
        '=== L3 高次洞察（最優先） ===\n' + l3_ctx + '\n\n'
        '=== L2 統合記憶 ===\n' + l2_ctx + '\n\n'
        '=== L1 生記憶（詳細補足） ===\n' + l1_ctx + '\n\n'
        '=== 質問 ===\n' + question
    )
    session_service = InMemorySessionService()
    agent = Agent(name='query_agent', model=get_llm(), instruction=QUERY_INSTRUCTION)
    return await run_agent(agent, session_service, prompt)


async def handle_status(request):
    return web.json_response(db_get_stats())

async def handle_memories(request):
    layer = int(request.rel_url.query.get('layer', '0'))
    if layer == 1:   return web.json_response({'layer': 1, 'data': db_get_l1()})
    elif layer == 2: return web.json_response({'layer': 2, 'data': db_get_l2()})
    elif layer == 3: return web.json_response({'layer': 3, 'data': db_get_l3()})
    else:            return web.json_response({'L1': db_get_l1(), 'L2': db_get_l2(), 'L3': db_get_l3()})

async def handle_ingest(request):
    body = await request.json()
    text = body.get('text', '').strip()
    if not text: return web.json_response({'error': 'text required'}, status=400)
    return web.json_response(await ingest_text(text, body.get('source', 'api')))

async def handle_query(request):
    q = request.rel_url.query.get('q', '').strip()
    if not q: return web.json_response({'error': 'q required'}, status=400)
    return web.json_response({'question': q, 'answer': await query_memory(q)})

async def handle_consolidate(request):
    layer = int(request.rel_url.query.get('layer', '0'))
    if layer == 2:   return web.json_response(await consolidate_l1())
    elif layer == 3: return web.json_response(await abstract_l2())
    else:
        r2 = await consolidate_l1()
        r3 = await abstract_l2()
        return web.json_response({'L1_to_L2': r2, 'L2_to_L3': r3})

async def handle_delete(request):
    body = await request.json()
    mid  = body.get('memory_id')
    if mid is None: return web.json_response({'error': 'memory_id required'}, status=400)
    db_delete_memory(int(mid))
    return web.json_response({'deleted': mid})

async def handle_clear(request):
    db_clear_all()
    return web.json_response({'status': 'cleared'})


def create_app():
    app = web.Application()
    app.router.add_get('/status',       handle_status)
    app.router.add_get('/memories',     handle_memories)
    app.router.add_post('/ingest',      handle_ingest)
    app.router.add_get('/query',        handle_query)
    app.router.add_post('/consolidate', handle_consolidate)
    app.router.add_post('/delete',      handle_delete)
    app.router.add_post('/clear',       handle_clear)
    return app


async def periodic_consolidate():
    while True:
        await asyncio.sleep(CONSOLIDATE_INTERVAL)
        ts = datetime.now().strftime('%H:%M:%S')
        print('[' + ts + '] 🔄 L1→L2 統合...')
        r = await consolidate_l1()
        print('  → ' + r.get('status') + ' (L1 ' + str(r.get('consolidated_l1_count', 0)) + ' 件)')


async def periodic_insight():
    await asyncio.sleep(INSIGHT_INTERVAL // 2)
    while True:
        await asyncio.sleep(INSIGHT_INTERVAL)
        ts = datetime.now().strftime('%H:%M:%S')
        print('[' + ts + '] 💡 L2→L3 洞察抽出...')
        r = await abstract_l2()
        print('  → ' + r.get('status') + ' (L2 ' + str(r.get('abstracted_l2_count', 0)) + ' 件)')


async def main(port=API_PORT):
    init_db()
    runner = web.AppRunner(create_app())
    await runner.setup()
    await web.TCPSite(runner, '0.0.0.0', port).start()
    print('🧠 Memory Agent (llama.cpp 3層版) 起動 → http://localhost:' + str(port))
    print('   llama-cpp-python API : ' + LLAMA_API_BASE)
    print('   L1→L2 統合間隔      : ' + str(CONSOLIDATE_INTERVAL) + '秒')
    print('   L2→L3 洞察間隔      : ' + str(INSIGHT_INTERVAL) + '秒')
    asyncio.create_task(periodic_consolidate())
    asyncio.create_task(periodic_insight())
    while True:
        await asyncio.sleep(3600)


if __name__ == '__main__':
    asyncio.run(main())


## ⑤ Memory Agent の起動

In [ ]:
import subprocess, time, requests, os

env = os.environ.copy()
env["LLAMA_API_BASE"]       = f"http://localhost:{LLAMA_PORT}/v1"
env["CONSOLIDATE_INTERVAL"] = "600"   # 10分（CPU 向け）

agent_proc = subprocess.Popen(
    ["python", "agent_llamacpp.py"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("🚀 Memory Agent を起動中...")
time.sleep(5)

for i in range(10):
    try:
        r = requests.get("http://localhost:8888/status", timeout=5)
        print(f"\n✅ Memory Agent 起動完了! ステータス: {r.json()}")
        break
    except:
        print(f"   待機中... ({i+1}/10)")
        time.sleep(4)
else:
    out = agent_proc.stdout.read(2000) if agent_proc.stdout else ""
    print(f"❌ 起動失敗。ログ:\n{out}")

## ⑥ Layer 1: 記憶の取り込み（Ingest）

IngestAgent がテキストを構造化し、**L1（生記憶）**として保存します。

In [ ]:
import requests, time

BASE_URL = "http://localhost:8888"

def ingest(text: str, source: str = "notebook"):
    t0 = time.time()
    r  = requests.post(f"{BASE_URL}/ingest", json={"text": text, "source": source}, timeout=120)
    result  = r.json()
    elapsed = time.time() - t0
    print(f"✅ 記憶 #{result.get('memory_id')} 保存 ({elapsed:.1f}秒)")
    print(f"   要約  : {result.get('summary')}")
    print(f"   重要度: {result.get('importance')}")
    print(f"   トピック: {result.get('topics')}")
    return result

print("=" * 55)
ingest(
    "llama-cpp-python は llama.cpp の Python バインディングです。"
    "pip install するだけで GGUF モデルを CPU で動かせ、"
    "OpenAI 互換の REST API サーバー機能も内蔵しています。",
    source="tech_doc"
)

print("=" * 55)
ingest(
    "GGUF は llama.cpp が採用するモデルファイル形式です。"
    "Q4_K_M などの量子化によりモデルサイズを大幅に削減でき、"
    "CPU のみで LLM を実行できるようになります。",
    source="tech_doc"
)

print("=" * 55)
ingest(
    "Google ADK (Agent Development Kit) は LiteLLM 経由で"
    "OpenAI 互換の任意のエンドポイントに接続できます。"
    "llama-cpp-python のサーバーとも組み合わせて使えます。",
    source="tech_doc"
)

## ⑦ Layer 2 & 3: 記憶の統合と洞察抽出（Consolidate）

2段階の統合を実行します：
- **L1→L2**: ConsolidateAgent が生記憶の関係性を見つけて統合記憶を生成
- **L2→L3**: InsightAgent が統合記憶をさらに抽象化して高次洞察を生成

In [ ]:
def show_layer_status():
    r = requests.get(f"{BASE_URL}/status")
    s = r.json()
    print("📊 メモリ階層の状況:")
    print(f"  L1 生記憶    : {s['L1_raw_memories']['total']} 件 "
          f"(未統合: {s['L1_raw_memories']['unconsolidated']} 件)")
    print(f"  L2 統合記憶  : {s['L2_consolidations']['total']} 件 "
          f"(未抽象化: {s['L2_consolidations']['unabstracted']} 件)")
    print(f"  L3 高次洞察  : {s['L3_insights']['total']} 件")

show_layer_status()

# --- Step 1: L1 → L2 統合 ---
print("\n🔄 Step1: L1→L2 統合 (ConsolidateAgent)...")
t0 = time.time()
r2 = requests.post(f"{BASE_URL}/consolidate", params={"layer": 2}, timeout=180).json()
print(f"  ステータス   : {r2.get('status')} ({time.time()-t0:.1f}秒)")
print(f"  統合したL1数 : {r2.get('consolidated_l1_count', 0)} 件")
print(f"  関係性       : {r2.get('connections', 'N/A')}")
print(f"  気づき       : {r2.get('insight', 'N/A')}")

# --- Step 2: L2 → L3 洞察 ---
print("\n💡 Step2: L2→L3 洞察抽出 (InsightAgent)...")
t0 = time.time()
r3 = requests.post(f"{BASE_URL}/consolidate", params={"layer": 3}, timeout=180).json()
print(f"  ステータス       : {r3.get('status')} ({time.time()-t0:.1f}秒)")
print(f"  抽象化したL2数   : {r3.get('abstracted_l2_count', 0)} 件")
print(f"  共通パターン     : {r3.get('connections', 'N/A')}")
print(f"  高次洞察         : {r3.get('insight', 'N/A')}")

print("")
show_layer_status()

## ⑧ QueryAgent: 3層すべてを参照して回答

QueryAgent は **L3→L2→L1 の優先順位**で記憶を参照し、回答の根拠を層ごとに引用します。

In [ ]:
def query(question: str):
    t0 = time.time()
    r  = requests.get(f"{BASE_URL}/query", params={"q": question}, timeout=180)
    result  = r.json()
    elapsed = time.time() - t0
    print(f"💬 質問: {result['question']}")
    print(f"\n🤖 回答 ({elapsed:.1f}秒):")
    print(result['answer'])
    return result

print("=" * 60)
query("llama-cpp-python と GGUF の関係を教えてください")

print("\n" + "=" * 60)
query("これらの技術を組み合わせると何ができますか？")

## ⑨ 3層メモリの一覧確認

In [ ]:
r = requests.get(f"{BASE_URL}/memories")  # 全層まとめて取得
data = r.json()

print("=" * 55)
print("📗 L1 生記憶")
for m in data.get("L1", []):
    flag = "✅" if m["consolidated"] else "🆕"
    print(f"  [{flag} L1-{m['id']}] 重要度:{m['importance']} | {m['summary']}")

print("\n📘 L2 統合記憶")
for m in data.get("L2", []):
    flag = "✅" if m["abstracted"] else "🆕"
    print(f"  [{flag} L2-{m['id']}] 関係性: {m['connections'][:80]}")
    print(f"           気づき : {m['insight'][:80]}")

print("\n📙 L3 高次洞察")
for m in data.get("L3", []):
    print(f"  [L3-{m['id']}] パターン: {m['connections'][:80]}")
    print(f"         洞察   : {m['insight'][:80]}")

print("\n")
show_layer_status()

## ⑩ 自由入力で試す

In [ ]:
my_text = "ここに覚えさせたいテキストを書いてください。"
ingest(my_text, source="user_input")

In [ ]:
my_question = "ここに質問を書いてください。"
query(my_question)

## ⑪ (オプション) GPU アクセラレーション

Colab の T4 GPU ランタイムを使う場合は、③のセルで `N_GPU_LAYERS` を変更してサーバーを再起動してください。

In [ ]:
# T4 GPU (15GB VRAM) の場合: llama-cpp-python を CUDA 対応でインストールし直す
# 実行する場合はコメントアウトを外してください

# import subprocess
# subprocess.run([
#     "pip", "install", "--upgrade", "--force-reinstall",
#     "llama-cpp-python[server]",
#     "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
# ], check=True)
#
# # N_GPU_LAYERS を設定してサーバーを再起動する
# server_env["N_GPU_LAYERS"] = "32"  # 1.5B モデルは全レイヤーをGPUに乗せられる
# llama_proc.terminate()
# # → セル③を再実行してサーバーを再起動してください

print("ℹ️ GPU を使う場合はコメントを外して実行し、セル③を再実行してください。")

## ⑫ (オプション) ngrok で外部公開

In [ ]:
NGROK_AUTH_TOKEN = ""  # https://dashboard.ngrok.com/get-started/your-authtoken

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    public_url = ngrok.connect(8888)
    print(f"🌐 外部公開 URL: {public_url}")
else:
    print("ℹ️ NGROK_AUTH_TOKEN を設定すると外部公開できます。")

## ⑬ API リファレンス（3層対応版）

| エンドポイント | メソッド | 説明 |
|---|---|---|
| `/status` | GET | **3層別**の件数統計を返す |
| `/memories` | GET | 全3層の記憶を返す。`?layer=1/2/3` で層を絞れる |
| `/ingest` | POST | L1に生記憶を追加 `{"text": "...", "source": "..."}` |
| `/query?q=...` | GET | **L3→L2→L1 優先**で3層を参照して回答 |
| `/consolidate` | POST | `?layer=2` でL1→L2のみ、`?layer=3` でL2→L3のみ、省略で両方 |
| `/delete` | POST | L1記憶を削除 `{"memory_id": 1}` |
| `/clear` | POST | 全層リセット |

## ⑭ ローカル環境での実行

```bash
# 1. 依存パッケージのインストール
pip install "llama-cpp-python[server]" huggingface_hub google-adk litellm aiohttp

# 2. GGUF モデルのダウンロード
huggingface-cli download Qwen/Qwen2.5-1.5B-Instruct-GGUF \
  qwen2.5-1.5b-instruct-q4_k_m.gguf --local-dir ./models

# 3. llama-cpp-python サーバーの起動
MODEL=./models/qwen2.5-1.5b-instruct-q4_k_m.gguf \
CHAT_FORMAT=chatml N_THREADS=8 N_CTX=2048 \
  python -m llama_cpp.server

# 4. Memory Agent の起動 (別ターミナル)
LLAMA_API_BASE=http://localhost:8000/v1 \
CONSOLIDATE_INTERVAL=600 INSIGHT_INTERVAL=1800 \
  python agent_llamacpp.py

# 5. テスト: 取り込み → 統合 → 洞察 → 質問
curl -X POST http://localhost:8888/ingest \
  -H 'Content-Type: application/json' \
  -d '{"text": "覚えさせたい情報", "source": "test"}'

curl -X POST 'http://localhost:8888/consolidate?layer=2'  # L1→L2
curl -X POST 'http://localhost:8888/consolidate?layer=3'  # L2→L3
curl 'http://localhost:8888/query?q=何を知っていますか'   # 3層参照
curl 'http://localhost:8888/memories?layer=3'             # L3洞察一覧
```

### Apple Silicon (M1/M2/M3) での Metal GPU 高速化
```bash
CMAKE_ARGS="-DGGML_METAL=on" pip install --upgrade --force-reinstall llama-cpp-python[server]
# → N_GPU_LAYERS=99 でほぼ全レイヤーをGPUで処理できる
```

### NVIDIA GPU (CUDA) での高速化
```bash
CMAKE_ARGS="-DGGML_CUDA=on" pip install --upgrade --force-reinstall llama-cpp-python[server]
# → N_GPU_LAYERS=32 (1.5B) ～ 99 (全レイヤー) を設定
```